# 위험 전환 예측 모델 6종 비교

**EDA 노트북(`colab_eda_only.ipynb`)의 결론을 사양으로 받아 구현한다.**

## 타겟
현재 **정상·주의** 인 사람이 → **다음 분기에 위험으로 전환**되는가 (0/1)

- 이미 위험인 사람은 학습 대상에서 **제외**. 이미 아픈 사람 맞히는 건 임상 가치가 없다.
- 같은 시점 `grade` 를 맞히면 **라벨 누출**(grade는 지표 합산으로 만든 값).
- 그냥 "다음 분기 위험 여부"를 맞히면 전이 관성(위험→위험 75.5%) 때문에
  모델이 **"지금 나쁘면 다음에도 나쁘다"** 만 학습한다.

## EDA에서 확정한 사양
| 항목 | 결정 | 근거 |
|---|---|---|
| 연속 특성 | 12개 + 나이 | 전환상관 0.05 이상 |
| 제외 | 혈색소·크레아티닌 | 상관 0.039 / 0.035 |
| 이진 특성 | 흡연·성별 | **비율비** 1.5배 이상 (상관계수로 자르면 안 됨) |
| 클리핑 | 혈압·혈당·중성지방·간수치 | 주입된 측정오류(혈압 589 등) 제거 |
| 로그변환 | 중성지방·GGT·AST·ALT | 로그정규 분포 |
| `d1` 보정 | gap으로 나눔 | 미수검으로 9.5%가 시간단위 오염 |
| `gap` 자체 | 특성 제외 | 전환율 편차 0.2%p — 예측력 없음 |
| `severity` | 특성 제외 | 생성기 내부 점수. 실제 병원에 없음 |
| 분할 | GroupShuffleSplit | 사람 단위. 겹치면 누출 |
| 평가 | **PR-AUC 주력** | 양성률 9.8% 불균형 |
| 하위집단 | 정상 출발 / 주의 출발 분리 | 양성의 91%가 주의 출신 |

## 0. 환경 준비

In [ ]:
!pip -q install lightgbm xgboost pyarrow
!apt-get install -y fonts-nanum > /dev/null 2>&1

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, warnings, time
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rc('font', family='NanumGothic'); plt.rc('axes', unicode_minus=False)
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font='NanumGothic')
pd.set_option('display.width', 200)
print('준비 완료')

## 1. 데이터 로드

In [ ]:
import os
PATH = '/content/checkups.parquet'   # 코랩 왼쪽 [파일] 탭에 올린 위치
assert os.path.exists(PATH), '파일 없음. [파일] 탭에 checkups.parquet 업로드 후 재실행.'

df = pd.read_parquet(PATH)
print(f'{len(df):,} 행 · {df.person_id.nunique():,} 명 · {df.quarter.nunique()} 분기')

KOR = {'bmi':'체질량지수','waist':'허리둘레','systolic':'수축기혈압','diastolic':'이완기혈압',
       'fbs':'공복혈당','total_chol':'총콜레스테롤','triglyceride':'중성지방','hdl':'HDL좋은콜',
       'ldl':'LDL나쁜콜','hemoglobin':'혈색소','ast':'간수치AST','alt':'간수치ALT',
       'ggt':'감마지티피','creatinine':'크레아티닌','age':'나이','grade':'종합판정'}
def kn(c):
    for sfx, lab in [('_lag1','·직전값'), ('_d1','·변화량'), ('_ma4','·1년평균'),
                     ('_std4','·변동성'), ('_dev','·평균대비')]:
        if c.endswith(sfx):
            return KOR.get(c[:-len(sfx)], c[:-len(sfx)]) + lab
    return {'is_male':'남성', 'smoker_i':'흡연', 'grade_lag1':'종합판정·직전값'}.get(c, KOR.get(c, c))

## 2. 전처리 + 특성 공학 + 타겟 생성

In [ ]:
# ---- EDA에서 확정한 사양 ----
METRICS_ALL = ['bmi','waist','systolic','diastolic','fbs','total_chol',
               'triglyceride','hdl','ldl','hemoglobin','ast','alt','ggt','creatinine']
DROP_METRICS = ['hemoglobin', 'creatinine']            # 전환상관 0.05 미만
USE_METRICS  = [m for m in METRICS_ALL if m not in DROP_METRICS]

CLIP = {'systolic': (70, 250), 'diastolic': (40, 150), 'fbs': (50, 400),
        'triglyceride': (20, 1000), 'ggt': (5, 500), 'ast': (5, 300),
        'alt': (3, 300), 'total_chol': (80, 400), 'bmi': (12, 50)}
LOGCOLS = ['triglyceride', 'ggt', 'ast', 'alt']        # 로그정규 분포
KEY     = ['bmi','systolic','diastolic','fbs','ldl','hdl','triglyceride','waist','ggt']
SUFFIX  = ['_lag1', '_d1', '_ma4', '_std4', '_dev']


def build_dataset(base):
    # 전처리 -> 파생특성 -> 타겟.
    # 파생은 반드시 행 필터링 '전에' 계산한다 (rolling 창이 깨지지 않도록).
    s = base.sort_values(['person_id','quarter']).copy()

    # (1) 생리학적 불가값 클리핑 -> 로그변환 (순서 중요)
    for c, (lo, hi) in CLIP.items():
        s[c] = s[c].clip(lo, hi)
    for c in LOGCOLS:
        s[c] = np.log1p(s[c])

    # (2) 파생특성. gap 으로 나눠 '분기당 변화율'로 시간단위 통일
    g = s.groupby('person_id')
    s['gap'] = s.quarter - g['quarter'].shift(1)
    for c in KEY:
        s[f'{c}_lag1'] = g[c].shift(1)
        s[f'{c}_d1']   = (s[c] - s[f'{c}_lag1']) / s['gap']       # <- gap 보정
        s[f'{c}_ma4']  = g[c].transform(lambda x: x.rolling(4, min_periods=1).mean())
        s[f'{c}_std4'] = g[c].transform(lambda x: x.rolling(4, min_periods=2).std())
        s[f'{c}_dev']  = s[c] - s[f'{c}_ma4']
    s['grade_lag1'] = g['grade'].shift(1)
    s['is_male']    = (s.sex == 'M').astype(int)
    s['smoker_i']   = s.smoker.astype(int)

    # (3) 다음 분기 판정을 붙이고 타겟 생성
    nx = base[['person_id','quarter','grade']].rename(columns={'grade':'next_grade'}).copy()
    nx['quarter'] -= 1
    s = s.merge(nx, on=['person_id','quarter'], how='inner')

    s = s[s.grade < 2].copy()                     # 이미 위험인 사람 제외
    s['target'] = (s.next_grade == 2).astype(int)
    return s


DERIVED = [f'{c}{sfx}' for c in KEY for sfx in SUFFIX]
FEATS   = USE_METRICS + DERIVED + ['age', 'is_male', 'smoker_i', 'grade', 'grade_lag1']
# gap: 예측력 없어서 제외 (d1 보정용으로만 사용) / severity: 생성기 내부값이라 제외
assert not {'gap','severity','severity_lag1','next_grade','target'} & set(FEATS)
print(f'특성 {len(FEATS)}개 = 원지표 {len(USE_METRICS)} + 파생 {len(DERIVED)} + 인적/판정 5')

## 3. 분석 대상 3만 명 샘플 (사람 단위)

In [ ]:
N_PEOPLE = 30_000
rng = np.random.default_rng(42)
sample_ids = rng.choice(df.person_id.unique(), N_PEOPLE, replace=False)
sdf = df[df.person_id.isin(sample_ids)].copy()

t0 = time.time()
s = build_dataset(sdf)
print(f'전처리 {time.time()-t0:.1f}s')
print(f'학습 대상 {len(s):,}행 · {s.person_id.nunique():,}명')
print(f'양성(다음 분기 위험 전환) {s.target.mean()*100:.2f}%  ({s.target.sum():,}개)')
print()
t = s.groupby('grade').target.agg(행수='size', 양성='sum', 전환율='mean')
t['전환율'] = (t['전환율']*100).round(2)
t.index = ['정상 출발','주의 출발']
print(t.to_string())
print(f'\n양성의 {t.loc["주의 출발","양성"]/t["양성"].sum()*100:.1f}%가 주의 출발')
print('-> 단일 모델은 사실상 주의 집단을 학습한다. 섹션 7에서 하위집단별로 분리 평가.')

## 4. 사람 단위 분할 (누출 방지)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

X = s[FEATS]; y = s['target'].values; groups = s['person_id'].values
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)
tr, te = next(gss.split(X, y, groups))
Xtr, Xte, ytr, yte = X.iloc[tr], X.iloc[te], y[tr], y[te]
gtr, gte = s.grade.values[tr], s.grade.values[te]

print(f'train {len(Xtr):,}행 / {pd.Series(groups[tr]).nunique():,}명 · 양성 {ytr.mean()*100:.2f}%')
print(f'test  {len(Xte):,}행 / {pd.Series(groups[te]).nunique():,}명 · 양성 {yte.mean()*100:.2f}%')
overlap = len(set(groups[tr]) & set(groups[te]))
print(f'겹치는 사람 {overlap}명  ->  {"정상" if overlap == 0 else "!!! 누출 !!!"}')

## 5. 모델 6종 학습·비교

평가는 **PR-AUC 주력**. 양성률 9.8% 불균형이라 AUC는 낙관적으로 나온다
(AUC 기준선 0.5 고정 / PR-AUC 기준선 = 양성률).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve)
import lightgbm as lgb, xgboost as xgb

MODELS = [
    ('LogisticRegression', make_pipeline(SimpleImputer(strategy='median'),
                                         StandardScaler(),
                                         LogisticRegression(max_iter=1000)), False),
    ('DecisionTree(CART)', DecisionTreeClassifier(max_depth=8, min_samples_leaf=50,
                                                  random_state=0), True),
    ('RandomForest',       RandomForestClassifier(n_estimators=300, max_depth=14,
                                                  min_samples_leaf=20, n_jobs=-1,
                                                  random_state=0), True),
    ('HistGradientBoosting', HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05,
                                                            random_state=0), False),
    ('LightGBM', lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                                    n_jobs=-1, random_state=0, verbose=-1), False),
    ('XGBoost',  xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                                   n_jobs=-1, random_state=0, tree_method='hist',
                                   eval_metric='logloss'), False),
]

BASELINE = yte.mean()
rows, fitted = [], {}
for name, model, needs_impute in MODELS:
    m = make_pipeline(SimpleImputer(strategy='median'), model) if needs_impute else model
    t0 = time.time(); m.fit(Xtr, ytr); fit_s = time.time() - t0
    prob = m.predict_proba(Xte)[:, 1]
    rows.append({'모델': name,
                 'PR-AUC': average_precision_score(yte, prob),
                 'AUC': roc_auc_score(yte, prob),
                 'PR-AUC/기준선': average_precision_score(yte, prob) / BASELINE,
                 '학습(초)': round(fit_s, 1)})
    fitted[name] = (m, prob)
    print(f'  {name:22} PR-AUC={rows[-1]["PR-AUC"]:.4f}  AUC={rows[-1]["AUC"]:.4f}  {fit_s:5.1f}s')

res = pd.DataFrame(rows).sort_values('PR-AUC', ascending=False).reset_index(drop=True)
print(f'\n기준선(양성률) PR-AUC = {BASELINE:.4f}')
print(res.round(4).to_string(index=False))
BEST = res.iloc[0]['모델']
print(f'\n최고: {BEST}  (PR-AUC {res.iloc[0]["PR-AUC"]:.4f} = 기준선의 {res.iloc[0]["PR-AUC/기준선"]:.1f}배)')

## 6. ROC / PR 곡선 · 성능 비교

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
for name, (_, prob) in fitted.items():
    fpr, tpr, _ = roc_curve(yte, prob)
    ax[0].plot(fpr, tpr, lw=1.4, label=f'{name} ({roc_auc_score(yte,prob):.3f})')
    pr, rc, _ = precision_recall_curve(yte, prob)
    ax[1].plot(rc, pr, lw=1.4, label=f'{name} ({average_precision_score(yte,prob):.3f})')
ax[0].plot([0,1],[0,1],'k--',lw=.8); ax[0].set_title('ROC 곡선')
ax[0].set_xlabel('거짓양성률'); ax[0].set_ylabel('재현율'); ax[0].legend(fontsize=8)
ax[1].axhline(BASELINE, ls='--', c='k', lw=.8, label=f'기준선 {BASELINE:.3f}')
ax[1].set_title('Precision-Recall 곡선 (불균형에서 이쪽이 정직)')
ax[1].set_xlabel('재현율'); ax[1].set_ylabel('정밀도'); ax[1].legend(fontsize=8)

r = res.set_index('모델')[['PR-AUC','AUC']]
r.plot(kind='barh', ax=ax[2]); ax[2].axvline(BASELINE, ls='--', c='crimson', lw=1)
ax[2].set_title('모델별 성능'); ax[2].set_xlabel('점수')
plt.tight_layout(); plt.show()

## 7. ★ 하위집단별 분리 평가

양성의 91%가 주의 출발이라, 합친 점수는 **주의 집단 성능에 지배당한다.**
정상 출발(희귀 급변)과 주의 출발(점진 악화)은 다른 문제이므로 따로 본다.

In [ ]:
_, best_prob = fitted[BEST]
rows = []
for gv, nm in [(0,'정상 출발'), (1,'주의 출발'), (None,'전체')]:
    mask = np.ones(len(yte), bool) if gv is None else (gte == gv)
    yy, pp = yte[mask], best_prob[mask]
    rows.append({'집단': nm, '행수': mask.sum(), '양성률': yy.mean(),
                 'PR-AUC': average_precision_score(yy, pp),
                 'AUC': roc_auc_score(yy, pp)})
sub = pd.DataFrame(rows)
sub['PR-AUC/기준선'] = (sub['PR-AUC'] / sub['양성률']).round(2)
sub['양성률'] = (sub['양성률']*100).round(2)
print(f'{BEST} 하위집단별 성능')
print(sub.round(4).to_string(index=False))
print()
print('해석: PR-AUC/기준선 이 클수록 무작위 대비 잘 잡는다는 뜻.')
print('      정상 출발은 양성률이 낮아 PR-AUC 절대값은 작게 나오는 게 정상이다.')
print('      두 집단의 배수를 비교해야 공정하다.')

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for gv, nm, c in [(0,'정상 출발','#2980b9'), (1,'주의 출발','#c0392b')]:
    mask = gte == gv
    pr, rc, _ = precision_recall_curve(yte[mask], best_prob[mask])
    ax[0].plot(rc, pr, lw=1.6, color=c, label=nm)
    ax[0].axhline(yte[mask].mean(), ls='--', lw=.8, color=c)
ax[0].set_title(f'{BEST} — 집단별 PR 곡선 (점선 = 각 기준선)')
ax[0].set_xlabel('재현율'); ax[0].set_ylabel('정밀도'); ax[0].legend()
sub[sub.집단!='전체'].set_index('집단')['PR-AUC/기준선'].plot(kind='bar', ax=ax[1], color=['#2980b9','#c0392b'])
ax[1].axhline(1, ls='--', c='k', lw=.8); ax[1].set_title('무작위 대비 배수'); ax[1].set_xlabel('')
ax[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## 8. 특성 중요도 (최고 성능 모델)

In [ ]:
best_model = fitted[BEST][0]
core = best_model[-1] if hasattr(best_model, 'steps') else best_model

imp = None
if hasattr(core, 'feature_importances_'):
    imp = pd.Series(core.feature_importances_, index=FEATS)
elif hasattr(core, 'coef_'):
    imp = pd.Series(np.abs(core.coef_[0]), index=FEATS)

if imp is not None:
    top = imp.sort_values(ascending=False).head(25)
    plt.figure(figsize=(9, 8))
    plt.barh([kn(i) for i in top.index][::-1], top.values[::-1], color='#34495e')
    plt.title(f'{BEST} — 특성 중요도 상위 25'); plt.xlabel('중요도')
    plt.tight_layout(); plt.show()

    # 파생 유형별로 묶어서 어떤 종류의 정보가 기여했는지 확인
    def kind(c):
        for sfx, lab in [('_lag1','직전값'), ('_d1','변화량'), ('_ma4','1년평균'),
                         ('_std4','변동성'), ('_dev','평균대비이탈')]:
            if c.endswith(sfx): return lab
        return '인적/판정' if c in ('age','is_male','smoker_i','grade','grade_lag1') else '현재값'
    grp = imp.groupby([kind(c) for c in imp.index]).sum().sort_values(ascending=False)
    print('=== 특성 유형별 기여도 합계 ===')
    print((grp/grp.sum()*100).round(1).to_string())
    print('\n변화량·변동성·평균대비이탈 비중이 크면 시계열 설계가 값을 했다는 뜻.')
else:
    print(f'{BEST} 는 중요도를 직접 제공하지 않음')

## 9. 운영 임계값 선택

확률을 그대로 쓰지 않고, **"상위 N%만 개입한다"** 는 운영 제약으로 임계값을 정한다.
병원 인력이 한정되어 있으므로 재현율보다 **정밀도(불러낸 사람 중 실제 전환자 비율)** 가 중요하다.

In [ ]:
from sklearn.metrics import precision_score, recall_score

rows = []
for topk in [0.02, 0.05, 0.10, 0.15, 0.20, 0.30]:
    thr = np.quantile(best_prob, 1 - topk)
    pred = (best_prob >= thr).astype(int)
    prec, rec = precision_score(yte, pred), recall_score(yte, pred)
    rows.append({'개입비율': f'상위 {topk*100:.0f}%', '임계확률': round(thr, 3),
                 '대상자수': int(pred.sum()), '정밀도': round(prec, 3),
                 '재현율': round(rec, 3), '기준대비': round(prec/BASELINE, 2)})
op = pd.DataFrame(rows)
print(f'무작위로 뽑으면 정밀도 = 양성률 = {BASELINE:.3f}')
print(op.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.5))
x = [r['개입비율'] for r in rows]
ax.plot(x, [r['정밀도'] for r in rows], marker='o', label='정밀도(불러낸 사람 중 적중)')
ax.plot(x, [r['재현율'] for r in rows], marker='s', label='재현율(전환자 중 포착)')
ax.axhline(BASELINE, ls='--', c='crimson', lw=1, label=f'무작위 정밀도 {BASELINE:.3f}')
ax.set_title('개입 규모별 정밀도·재현율 트레이드오프'); ax.set_ylabel('비율'); ax.legend()
plt.tight_layout(); plt.show()

## 10. 학습곡선 — 3만 명 샘플이 충분한지 검증

In [ ]:
sizes = [3_000, 6_000, 12_000, 20_000, 30_000]
curve = []
for n in sizes:
    ids = rng.choice(sample_ids, n, replace=False)
    ss = build_dataset(df[df.person_id.isin(ids)])
    Xs, ys, gs = ss[FEATS], ss.target.values, ss.person_id.values
    a, b = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0).split(Xs, ys, gs))
    m = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                           n_jobs=-1, random_state=0, verbose=-1)
    m.fit(Xs.iloc[a], ys[a])
    p = m.predict_proba(Xs.iloc[b])[:, 1]
    curve.append({'인원': n, '행수': len(ss),
                  'PR-AUC': average_precision_score(ys[b], p),
                  'AUC': roc_auc_score(ys[b], p)})
    print(f'  {n:>6,}명  PR-AUC={curve[-1]["PR-AUC"]:.4f}  AUC={curve[-1]["AUC"]:.4f}')

cv = pd.DataFrame(curve)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(cv.인원, cv['PR-AUC'], marker='o', label='PR-AUC')
ax.plot(cv.인원, cv['AUC'], marker='s', label='AUC')
ax.set_xlabel('학습 인원'); ax.set_ylabel('점수'); ax.set_title('학습곡선 (LightGBM)')
ax.legend(); plt.tight_layout(); plt.show()

gain = cv['PR-AUC'].iloc[-1] - cv['PR-AUC'].iloc[-2]
print(f'\n마지막 구간 PR-AUC 증가폭 {gain:+.4f}')
print('증가폭이 거의 0이면 3만 명으로 충분. 아직 오르면 10만 전체를 쓰는 게 낫다.')

## 11. 모델 저장 (API 연결용)

In [ ]:
import joblib, json

best_model = fitted[BEST][0]
joblib.dump({'model': best_model, 'features': FEATS,
             'clip': CLIP, 'logcols': LOGCOLS, 'key': KEY, 'suffix': SUFFIX,
             'baseline': float(BASELINE), 'best_name': BEST},
            '/content/risk_model.joblib')

meta = {
    'target': '현재 정상·주의 -> 다음 분기 위험 전환',
    'best_model': BEST,
    'pr_auc': float(res.iloc[0]['PR-AUC']),
    'auc': float(res.iloc[0]['AUC']),
    'baseline': float(BASELINE),
    'n_people': int(N_PEOPLE),
    'n_rows': int(len(s)),
    'n_features': len(FEATS),
    'subgroup': sub.to_dict('records'),
}
with open('/content/risk_model_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(json.dumps(meta, ensure_ascii=False, indent=2))
print('\n저장 완료. 코랩 [파일] 탭에서 risk_model.joblib 을 내려받아')
print('health-log-api/models/ 에 넣으면 API에서 불러 쓸 수 있다.')

## 12. 요약

In [ ]:
print('='*72); print('모델링 요약'); print('='*72)
print(f'타겟      현재 정상·주의 -> 다음 분기 위험 전환')
print(f'학습      {N_PEOPLE:,}명 · {len(s):,}행 · 특성 {len(FEATS)}개 · 양성률 {s.target.mean()*100:.2f}%')
print()
print('모델 6종 (PR-AUC 내림차순)')
print(res.round(4).to_string(index=False))
print()
print(f'최고 {BEST}  PR-AUC {res.iloc[0]["PR-AUC"]:.4f}'
      f'  (무작위 {BASELINE:.4f} 의 {res.iloc[0]["PR-AUC/기준선"]:.1f}배)')
print()
print('하위집단'); print(sub.round(4).to_string(index=False))
print()
print('운영 임계값'); print(op.to_string(index=False))
print('='*72)